# Drill 1: GRPO advantages and policy losses (25 min)

Blank page, no AI, docs are fine. Fill the two cells marked `# TODO`, then run the test cells. Each prints `ok`.

## Spec

**`grpo_advantages(rewards, G)`**: `rewards` is `(N*G,)`, consecutive blocks of `G` come from one prompt.
`A = (r - group_mean) / (group_std + eps)` with the population std (`correction=0`). A group whose rewards are all
equal must give `A = 0`, not NaN.

**`policy_loss(logp_new, logp_old, A, mask, mode, ...)`**, with `r = exp(logp_new - logp_old)` per token and `sg` = stop-gradient (`.detach()`):

| mode | per-token loss `l_t` |
|---|---|
| `reinforce` | `-A * logp_new` |
| `ppo` | `-min(r * A, clip(r, 1 - eps_lo, 1 + eps_hi) * A)` |
| `cispo` | `-sg(clip(r, 0, cispo_max)) * A * logp_new` |

Then aggregate over real tokens with `norm="token"` (sum / total real tokens) or `norm="seq"` (mean within each
response, then mean over responses).

`gspo` is one ratio per response: `s = exp(mean over the response's real tokens of (logp_new - logp_old))`,
`l = -min(s * A, clip(s, 1 - eps_lo, 1 + eps_hi) * A)`, loss = mean over responses (ignore `norm`).
(The GSPO paper uses a much tighter band; we reuse PPO's to keep the tests readable.)

**Watch out:** padded positions of `logp_old` can hold garbage like `-100`. `exp(100)` is `inf` in float32 and
`inf * 0 = nan`, so multiplying by the mask at the end is not enough.

In [1]:
import torch

In [2]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
G = 4
rewards = torch.tensor([1., 0, 0, 0,   1, 1, 1, 1,   0, 1, 0, 1])   # (N*G,), three prompts

In [32]:
r = rewards.view(G, -1) # [4,3]
m = r.mean(axis=-1).unsqueeze(-1)
s = r.std(axis=-1).unsqueeze(-1)
r.shape, m.shape
#r_grpo = (r - r.mean(axis=-1)) / (r.std(axis=-1)  + eps)
(r - m) (s + eps)

tensor([[ 0.6667, -0.3333, -0.3333],
        [-0.6667,  0.3333,  0.3333],
        [ 0.3333,  0.3333, -0.6667],
        [ 0.3333, -0.6667,  0.3333]])

In [37]:
def grpo_advantages(rewards, G, eps=1e-6):
    """rewards (N*G,): consecutive blocks of G rewards come from the same prompt.
    Returns A (N*G,) = (r - group mean) / (group std + eps), std with correction=0 (population std)."""
    r = rewards.view(G, -1) # [4,3]
    m = r.mean(axis=-1).unsqueeze(-1)
    s = r.std(axis=-1).unsqueeze(-1)
    #r_grpo = (r - r.mean(axis=-1)) / (r.std(axis=-1)  + eps)
    return ((r - m) / (s + eps))


In [38]:
grpo_advantages(rewards, G)

tensor([[ 1.1547, -0.5773, -0.5773],
        [-1.1547,  0.5773,  0.5773],
        [ 0.5773,  0.5773, -1.1547],
        [ 0.5773, -1.1547,  0.5773]])

## The modes

`r = p_new / p_old`: how far the policy moved on a token since sampling.

- **reinforce**: `-A * log p`. Ignores `r`. Baseline.
- **ppo**: weight by `r`, but zero the gradient once a token moved far enough in the direction `A` wants. Know from memory.
- **cispo**: never zero a token, just cap the weight. Weight is detached.
- **gspo**: one ratio per response instead of per token; clip the whole response together.

In [39]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
logp_new = torch.log(torch.tensor([[0.5, 0.4, 0.3],
                                   [0.2, 0.6, 0.9]])).requires_grad_()   # (B, T) = (2, 3)
logp_old = torch.log(torch.tensor([[0.4, 0.4, 0.6],
                                   [0.2, 0.3, 0.9]]))                    # (B, T)
logp_old[1, 2] = -100.0                                                  # garbage in the padded slot
A = torch.tensor([1.0, -1.0])                                            # (B,)
mask = torch.tensor([[1., 1., 1.],
                     [1., 1., 0.]])                                      # (B, T)

In [78]:
(logp_new - logp_old) * mask

tensor([[ 0.2231,  0.0000, -0.6931],
        [ 0.0000,  0.6931,  0.0000]], grad_fn=<MulBackward0>)

In [79]:
log_masked_mean = ((logp_new - logp_old) * mask).sum(axis=-1) / mask.sum(axis=-1)
log_masked_mean
#w.mean(axis=-1).unsqueeze(-1)

tensor([-0.1567,  0.3466], grad_fn=<DivBackward0>)

In [99]:
def policy_loss(logp_new, logp_old, A, mask, mode, eps_lo=0.2, eps_hi=0.28, cispo_max=4.0, norm="token"):
    """
    logp_new (B, T)  log-prob of the sampled tokens under the current policy (has grad)
    logp_old (B, T)  same tokens under the rollout policy (no grad). Padded positions may hold garbage.
    A        (B,)    one advantage per response
    mask     (B, T)  1 = real token, 0 = padding. Every row has at least one real token.
    mode     "reinforce" | "ppo" | "cispo" | "gspo"
    norm     "token": sum over real tokens / number of real tokens in the batch
             "seq":   per response, mean over its real tokens; then mean over responses
    Returns a scalar loss (minimize it).
    """
    # w x A x logp / norm
    A = A.unsqueeze(-1) # [B, 1]
    # flip the logp for loss. * minimizing the loss * 
    masked = - logp_new * mask # [B, T]

    w = torch.exp((logp_new - logp_old) * mask ).detach()

    if mode == 'gspo' and norm == 'token':
        raise ValueError()

    if mode == 'reinforce':
        w = torch.ones_like(masked)
    elif mode == 'ppo':
        w[(A > 0) & (w > 1 + eps_hi)] = 0
        w[(A < 0) & (w < 1 - eps_lo)] = 0
    elif mode == 'cispo':
        # just cap it
        w = torch.clip(w, 0, cispo_max)
    elif mode == 'gspo':
        # one ratio per reponse. clips per sequence
        w = (((logp_new - logp_old) * mask).sum(axis=-1) / mask.sum(axis=-1)).exp() # [B]
        w = w.unsqueeze(-1).detach() # [B, 1]
        w[(A > 0) & (w > 1 + eps_hi)] = 0
        w[(A < 0) & (w < 1 - eps_lo)] = 0
        
        
    x = w * A * masked # [B, T]

    if norm == 'token':
        return x.sum() / mask.sum()
    elif norm == 'seq':
        x_seq = x.sum(axis=-1) # [B]
        mask_seq = mask.sum(axis=-1) # [B]
        return (x_seq / mask_seq).mean()
        

In [95]:
l_token = policy_loss(logp_new, logp_old, A, mask, mode='reinforce', norm='token')
l_seq = policy_loss(logp_new, logp_old, A, mask, mode='reinforce', norm='seq')
l_token, l_seq

(tensor(0.1386, grad_fn=<DivBackward0>),
 tensor(-0.0612, grad_fn=<MeanBackward0>))

In [96]:
l_token = policy_loss(logp_new, logp_old, A, mask, mode='ppo', norm='token')
l_seq = policy_loss(logp_new, logp_old, A, mask, mode='ppo', norm='seq')
l_token, l_seq

(tensor(-0.0493, grad_fn=<DivBackward0>),
 tensor(-0.2603, grad_fn=<MeanBackward0>))

In [98]:
l_seq = policy_loss(logp_new, logp_old, A, mask, mode='gspo', norm='seq')
l_seq

tensor(-0.3487, grad_fn=<MeanBackward0>)

**Variant: from logits.** Same loss, but you get the raw logits and the chosen tokens `y`, and compute the log-probs yourself.

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
B, T, V = 2, 3, 5
logits = torch.randn(B, T, V, requires_grad=True)                    # (B, T, V)
y = torch.randint(V, (B, T))                                         # (B, T) chosen tokens

In [ ]:
def policy_loss_from_logits(logits, y, logp_old, A, mask, mode, **kw):
    """logits (B, T, V), y (B, T) chosen token ids; the rest as in policy_loss.
    Compute logp_new (B, T) from logits and y, then return policy_loss(...). Must stay finite for huge logits."""
    # TODO
    raise NotImplementedError


## Tests

Test helpers (run, don't edit).

In [ ]:
def make_batch(seed=0, B=4, T=5, V=7):
    g = torch.Generator().manual_seed(seed)
    logits = torch.randn(B, T, V, generator=g).requires_grad_()      # (B, T, V)
    y = torch.randint(V, (B, T), generator=g)                        # (B, T) sampled tokens
    mask = torch.ones(B, T)                                          # (B, T)
    mask[1, 3:] = 0
    mask[2, 1:] = 0
    A = torch.tensor([1.0, -0.5, 2.0, -1.0])                         # (B,)
    return logits, y, mask, A


def token_logp(logits, y):
    return torch.log_softmax(logits, -1).gather(-1, y.unsqueeze(-1)).squeeze(-1)   # (B, T)


def logits_grad(mode, norm="token", garbage=False, seed=0):
    """On-policy batch (logp_old = logp_new). Returns loss, dloss/dlogits (B, T, V), batch."""
    logits, y, mask, A = make_batch(seed)
    logp = token_logp(logits, y)
    logp_old = logp.detach().clone()
    if garbage:
        logp_old[mask == 0] = -100.0
    loss = policy_loss(logp, logp_old, A, mask, mode, norm=norm)
    loss.backward()
    return loss, logits.grad, (logits, y, mask, A)


def ratio_grad(mode, r, A, **kw):
    """T = 1, one token per response, with exp(logp_new - logp_old) = r. Returns dloss/dlogp_new (B,)."""
    logp_new = torch.full((len(r), 1), -1.0, requires_grad=True)    # (B, 1)
    logp_old = (-1.0 - torch.log(torch.tensor(r))).view(-1, 1)        # (B, 1)
    loss = policy_loss(logp_new, logp_old, torch.tensor(A), torch.ones(len(r), 1), mode, **kw)
    loss.backward()
    return logp_new.grad.view(-1)

**Test 1. GRPO advantages.** Rewards `[1,0,0,0]`, `[1,1,1,1]`, `[0,1,0,1]`.

In [ ]:
A = grpo_advantages(torch.tensor([1., 0, 0, 0,   1, 1, 1, 1,   0, 1, 0, 1]), G=4)
expected = torch.tensor([1.7321, -0.5774, -0.5774, -0.5774,   0, 0, 0, 0,   -1, 1, -1, 1])
assert A.shape == (12,), A.shape
assert torch.isfinite(A).all(), "an all-equal group must give A = 0, not NaN"
assert torch.allclose(A, expected, atol=1e-3), A
print("ok")

**Test 2. The logits gradient is `A * (p - onehot) / norm`.** This is the formula from `rl-corrections/corrections.md`.
`norm` is the total number of real tokens for `"token"`, and `B * len(response)` for `"seq"`.

In [ ]:
_, g_tok, (logits, y, mask, A) = logits_grad("reinforce", "token")
p = torch.softmax(logits.detach(), -1)                               # (B, T, V)
onehot = torch.nn.functional.one_hot(y, p.shape[-1]).float()         # (B, T, V)
dz = A.view(-1, 1, 1) * (p - onehot) * mask.unsqueeze(-1)            # (B, T, V)
assert torch.allclose(g_tok, dz / mask.sum(), atol=1e-6), "norm='token' is wrong"
_, g_seq, _ = logits_grad("reinforce", "seq")
assert torch.allclose(g_seq, dz / (mask.sum(-1).view(-1, 1, 1) * mask.shape[0]), atol=1e-6), "norm='seq' is wrong"
print("ok")

**Test 3. On-policy, every method is REINFORCE.** With `r = 1`, PPO and CISPO equal REINFORCE, and GSPO equals REINFORCE with per-response normalization.

In [ ]:
_, g_ref, _ = logits_grad("reinforce", "token")
for mode in ["ppo", "cispo"]:
    _, g, _ = logits_grad(mode, "token")
    assert torch.allclose(g, g_ref, atol=1e-6), f"{mode} on-policy should equal reinforce"
_, g_ref_seq, _ = logits_grad("reinforce", "seq")
_, g_gspo, _ = logits_grad("gspo")
assert torch.allclose(g_gspo, g_ref_seq, atol=1e-6), "on-policy gspo should equal reinforce with norm='seq'"
print("ok")

**Test 4. Garbage in the padding.** `logp_old = -100` at padded positions: loss and gradients stay finite, padded tokens get zero gradient, and the real tokens' gradient does not change.

In [ ]:
for mode in ["reinforce", "ppo", "cispo", "gspo"]:
    for norm in ["token", "seq"]:
        loss, g, (logits, y, mask, A) = logits_grad(mode, norm, garbage=True)
        assert torch.isfinite(loss), f"{mode}/{norm}: loss is {loss}"
        assert torch.isfinite(g).all(), f"{mode}/{norm}: non-finite gradient"
        assert (g[mask == 0] == 0).all(), f"{mode}/{norm}: padded tokens got gradient"
        _, g_clean, _ = logits_grad(mode, norm, garbage=False)
        assert torch.allclose(g, g_clean, atol=1e-6), f"{mode}/{norm}: padding garbage changed the gradient"
print("ok")

**Test 5. PPO clipping.** Four one-token responses, gradient w.r.t. `logp_new`, `norm="token"` (divide by 4).
Unclipped, `d(-r*A)/dlogp = -r*A`.

| r | A | what happens | grad |
|---|---|---|---|
| 1.5 | +1 | moved up past 1.28, stop | 0 |
| 1.5 | -1 | bad token got more likely, no cap | 1.5 / 4 = 0.375 |
| 0.5 | +1 | good token got less likely, no cap | -0.5 / 4 = -0.125 |
| 0.5 | -1 | moved down past 0.8, stop | 0 |

In [ ]:
g = ratio_grad("ppo", [1.5, 1.5, 0.5, 0.5], [1., -1., 1., -1.])
assert torch.allclose(g, torch.tensor([0.0, 0.375, -0.125, 0.0]), atol=1e-5), g
print("ok")

**Test 6. CISPO never drops a token, caps at 4, and detaches the weight.** Expected `-w * A / 4` with
`w = clip(r, 0, 4)`. Without `.detach()` the product rule adds a term, and the unclipped rows come out wrong.

In [ ]:
g = ratio_grad("cispo", [1.5, 10.0, 0.05, 1.0], [1., 1., -1., 1.])
assert torch.allclose(g, torch.tensor([-0.375, -1.0, 0.0125, -0.25]), atol=1e-5), g
print("ok")

**Test 7. GSPO clips the whole response.** Response 0 has `s = 1.5` (A > 0): every token gets 0, even
token 1, whose own ratio is `1.5 * e^-0.3 = 1.11` and would survive per-token PPO. Response 1 has `s = 0.9`, inside the band:
each token gets `-A * s / T / B = -0.9 / 3 / 2 = -0.15`.

In [ ]:
logp_new = torch.zeros(2, 3, requires_grad=True)                    # (B, T)
d0 = torch.log(torch.tensor(1.5)) + torch.tensor([0.3, -0.3, 0.0])     # mean log-ratio = log 1.5
d1 = torch.log(torch.tensor(0.9)).repeat(3)
logp_old = -torch.stack([d0, d1])                                     # so logp_new - logp_old = d
policy_loss(logp_new, logp_old, torch.tensor([1., 1.]), torch.ones(2, 3), "gspo").backward()
expected = torch.tensor([[0., 0., 0.], [-0.15, -0.15, -0.15]])
assert torch.allclose(logp_new.grad, expected, atol=1e-5), logp_new.grad
print("ok")

**Test 8. `policy_loss_from_logits`** matches `policy_loss` and stays finite for huge logits.

In [ ]:
for mode in ["reinforce", "ppo", "cispo", "gspo"]:
    logits, y, mask, A = make_batch()
    logp_old = token_logp(logits, y).detach()
    loss = policy_loss_from_logits(logits, y, logp_old, A, mask, mode)
    loss.backward()
    logits2 = logits.detach().clone().requires_grad_()
    loss2 = policy_loss(token_logp(logits2, y), logp_old, A, mask, mode)
    loss2.backward()
    assert torch.allclose(loss, loss2, atol=1e-6), mode
    assert torch.allclose(logits.grad, logits2.grad, atol=1e-6), mode

# huge logits: most probabilities underflow to 0 in float32
logits, y, mask, A = make_batch()
big = (logits.detach() * 200).requires_grad_()
loss = policy_loss_from_logits(big, y, torch.full_like(mask, -50.0), A, mask, "reinforce")
loss.backward()
assert torch.isfinite(loss) and torch.isfinite(big.grad).all(), "not numerically stable"
print("ok")